# Cyber Risk Premium Quotation Agent
## Agentic AI for Insurance Premium Generation

**Project:** Cyber & AI Risk Insurance Pricing  
**Phase:** 4 - Agentic AI Deployment  
**Date:** June 24, 2026  
**Framework:** Agno + Google Gemini  

## What this notebook does

Builds a complete cyber insurance premium quotation agent that:
1. Collects company information from users
2. Predicts incident frequency using a Poisson GLM model
3. Predicts loss severity using a Lognormal distribution
4. Calculates premiums with industry-standard loading factors
5. Generates **Tier 1** (Primary) and **Tier 2** (Secondary) coverage quotes
6. Provides detailed breakdowns and recommendations

### Coverage Tiers

**Tier 1: Primary Coverage (100% of base premium)**
- Ransomware attacks
- Data Breaches
- Supply Chain compromise

**Tier 2: Secondary Coverage (125% of base premium)**
- DDoS attacks
- Malware infections
- Trojans
- Backdoors
- Advanced Persistent Threats (APTs)

## Prerequisites

- Google account (for Colab)
- Google Gemini API key (free tier available at https://aistudio.google.com/apikey)
- Store API key in Colab Secrets as `GOOGLE_API_KEY`

## 1. Install dependencies

Install Agno framework and Google Gemini SDK

In [ ]:
!pip install -q agno google-genai
print("✓ Dependencies installed successfully")

## 2. Standard imports and configuration

In [ ]:
# === Standard imports ===
import os
import json
import numpy as np
import pandas as pd
from typing import Any, Dict
from dataclasses import dataclass
from datetime import datetime

# Agno framework
from agno.agent import Agent
from agno.models.google import Gemini

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Display options
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

print("Imports OK.")

## 3. Model Parameters and Coefficients

Load all pre-trained model coefficients for frequency, severity, and premium calculation

In [ ]:
@dataclass
class ModelParameters:
    """Frequency and Severity Model Parameters"""
    
    # Frequency Model (Poisson GLM with Ridge Regularization)
    frequency_intercept: float = -0.3326
    frequency_log_revenue: float = 0.0478
    frequency_log_employees: float = -0.0512
    frequency_is_public: float = -0.0094
    frequency_size: float = -0.0405
    
    # Severity Model (Lognormal Distribution)
    severity_intercept: float = 9.5316
    severity_log_revenue: float = 0.3586
    severity_log_employees: float = -0.0627
    severity_is_public: float = -0.1252
    severity_log_records: float = 0.0203
    severity_mu: float = 16.6899
    severity_sigma: float = 1.6415
    
    # Loading Factors (58% total)
    loading_acquisition: float = 0.20
    loading_admin: float = 0.10
    loading_profit: float = 0.15
    loading_uncertainty: float = 0.08
    loading_reinsurance: float = 0.05
    
    # Coverage Tier Adjustments
    tier1_coverage_factor: float = 1.00  # Base premium (100%)
    tier2_coverage_factor: float = 1.25  # Secondary coverage (+25%)
    
    @property
    def total_loading(self) -> float:
        return (self.loading_acquisition + self.loading_admin + 
                self.loading_profit + self.loading_uncertainty + 
                self.loading_reinsurance)
    
    @property
    def loading_multiplier(self) -> float:
        return 1 + self.total_loading


# Initialize parameters
PARAMS = ModelParameters()

print("✓ Model parameters loaded")
print(f"  • Frequency model: Poisson GLM with Ridge regularization")
print(f"  • Severity model: Lognormal distribution")
print(f"  • Total loading factor: {PARAMS.total_loading*100:.1f}%")
print(f"  • Loading multiplier: {PARAMS.loading_multiplier:.3f}")
print(f"  • Tier 1 coverage: 100% of base premium")
print(f"  • Tier 2 coverage: 125% of base premium")

## 4. Industry and Revenue Tier Data

Reference tables for industry and company size relativities

In [ ]:
# Industry Premium Relativities (vs. mean premium)
INDUSTRY_RELATIVITIES = {
    '51': 1.231,   # Technology
    '52': 1.181,   # Finance
    '44-45': 1.264,  # Retail
    '92': 1.221,   # Telecom
    '31-33': 1.023,  # Industrial
    '21': 0.955,   # Mining
    '62': 0.914,   # IT Services
    '22': 0.885,   # Utilities
    '42': 0.804,   # Services
    '55': 1.417,   # Specialty
}

# Company Size Categories
REVENUE_TIER_RANGES = {
    'Q1_Small': (0, 1e9),
    'Q2_Medium': (1e9, 10e9),
    'Q3_Large': (10e9, 100e9),
    'Q4_Enterprise': (100e9, float('inf'))
}

REVENUE_TIER_RELATIVITIES = {
    'Q1_Small': 0.346,
    'Q2_Medium': 0.578,
    'Q3_Large': 1.024,
    'Q4_Enterprise': 2.050
}

# Industry Code Reference
INDUSTRY_CODES = {
    '21': 'Mining',
    '22': 'Utilities',
    '31-33': 'Industrial Manufacturing',
    '42': 'Business Services',
    '44-45': 'Retail',
    '51': 'Technology',
    '52': 'Finance & Insurance',
    '55': 'Specialty',
    '62': 'IT Services',
    '92': 'Telecom',
}

print("✓ Industry and revenue tier data loaded")
print(f"  • Industry codes: {len(INDUSTRY_CODES)}")
print(f"  • Revenue tiers: {len(REVENUE_TIER_RANGES)}")
print(f"  • Industry relativities: {len(INDUSTRY_RELATIVITIES)}")

## 5. Core Prediction Functions

Frequency and severity prediction models

In [ ]:
def predict_frequency(company_revenue: float, 
                     employee_count: int,
                     is_public: bool) -> dict:
    """
    Predict incident frequency using Poisson GLM
    E[Frequency] = exp(β₀ + β₁·log_revenue + β₂·log_employees + ...)
    """
    
    # Feature transformation
    log_revenue = np.log1p(company_revenue)
    log_employees = np.log1p(employee_count)
    is_public_numeric = 1 if is_public else 0
    
    # Determine company size category
    if company_revenue < 1e9:
        size_category = 0  # Small
    elif company_revenue < 10e9:
        size_category = 1  # Medium
    elif company_revenue < 100e9:
        size_category = 2  # Large
    else:
        size_category = 3  # Enterprise
    
    # Poisson GLM prediction
    log_lambda = (PARAMS.frequency_intercept +
                  PARAMS.frequency_log_revenue * log_revenue +
                  PARAMS.frequency_log_employees * log_employees +
                  PARAMS.frequency_is_public * is_public_numeric +
                  PARAMS.frequency_size * size_category)
    
    predicted_frequency = np.exp(log_lambda)
    risk_score = (predicted_frequency / 1.1354) * 100
    
    return {
        'predicted_frequency': round(predicted_frequency, 4),
        'risk_score': round(risk_score, 1),
        'size_category': ['Small', 'Medium', 'Large', 'Enterprise'][size_category],
    }


def predict_severity(company_revenue: float,
                    employee_count: int,
                    is_public: bool,
                    records_at_risk: int = 1000000) -> dict:
    """
    Predict expected loss per incident using Severity Model
    log(Loss) = β₀ + β₁·log_revenue + β₂·log_employees + ...
    """
    
    # Feature transformation
    log_revenue = np.log1p(company_revenue)
    log_employees = np.log1p(employee_count)
    is_public_numeric = 1 if is_public else 0
    log_records = np.log1p(records_at_risk)
    
    # Ridge regression on log scale
    log_loss = (PARAMS.severity_intercept +
                PARAMS.severity_log_revenue * log_revenue +
                PARAMS.severity_log_employees * log_employees +
                PARAMS.severity_is_public * is_public_numeric +
                PARAMS.severity_log_records * log_records)
    
    expected_loss = np.exp(log_loss)
    
    return {
        'expected_severity': round(expected_loss, 0),
        'q95_loss': round(np.exp(PARAMS.severity_mu + PARAMS.severity_sigma * 1.6449), 0),
    }


def get_industry_relativity(industry_code: str) -> float:
    """Get industry premium relativity factor"""
    return INDUSTRY_RELATIVITIES.get(industry_code, 1.0)


def get_revenue_tier_relativity(company_revenue: float) -> tuple:
    """Get revenue tier and its relativity factor"""
    for tier, (min_rev, max_rev) in REVENUE_TIER_RANGES.items():
        if min_rev <= company_revenue < max_rev:
            return tier, REVENUE_TIER_RELATIVITIES[tier]
    return 'Unknown', 1.0

print("✓ Prediction functions defined")

## 6. Premium Calculation and Coverage Tier Functions

In [ ]:
def calculate_pure_premium(frequency: float, severity: float) -> dict:
    """
    Calculate pure premium (actuarial cost) = Frequency × Severity
    Then apply loading factors
    """
    
    pure_premium = frequency * severity
    
    # Loading components
    loading_dict = {
        'acquisition': pure_premium * PARAMS.loading_acquisition,
        'admin': pure_premium * PARAMS.loading_admin,
        'profit': pure_premium * PARAMS.loading_profit,
        'uncertainty': pure_premium * PARAMS.loading_uncertainty,
        'reinsurance': pure_premium * PARAMS.loading_reinsurance,
    }
    
    total_loading = sum(loading_dict.values())
    final_premium = pure_premium + total_loading
    
    return {
        'pure_premium': round(pure_premium, 0),
        'loading_components': {k: round(v, 0) for k, v in loading_dict.items()},
        'total_loading': round(total_loading, 0),
        'final_premium': round(final_premium, 0),
    }


def generate_coverage_tiers(final_premium: float) -> dict:
    """
    Generate Tier 1 and Tier 2 coverage options
    
    Tier 1: Primary Coverage - Ransomware, Data Breaches, Supply Chain (Base)
    Tier 2: Secondary Coverage - DDoS, Malware, Trojans, Backdoors, APTs (+25%)
    """
    
    tier1_premium = final_premium * PARAMS.tier1_coverage_factor
    tier2_premium = final_premium * PARAMS.tier2_coverage_factor
    combined_premium = tier1_premium + tier2_premium
    
    return {
        'tier1': {
            'name': 'Primary Coverage (Tier 1)',
            'attack_vectors': ['Ransomware', 'Data Breaches', 'Supply Chain'],
            'annual_premium': round(tier1_premium, 0),
            'percentage_of_base': '100%'
        },
        'tier2': {
            'name': 'Secondary Coverage (Tier 2)',
            'attack_vectors': ['DDoS', 'Malware', 'Trojans', 'Backdoors', 'APTs'],
            'annual_premium': round(tier2_premium, 0),
            'percentage_of_base': '125%',
            'note': 'Standalone or combined with Tier 1'
        },
        'combined': {
            'name': 'Complete Coverage (Tier 1 + Tier 2)',
            'attack_vectors': ['All cyber threats covered'],
            'annual_premium': round(combined_premium, 0),
            'recommendation': 'Recommended for enterprise and financial companies'
        }
    }

print("✓ Premium calculation functions defined")

## 7. Agent Tool Definitions

Define the tools that the agent will use to generate quotes

In [ ]:
def premium_quotation_tool(company_name: str,
                          company_revenue_usd: float,
                          employee_count: int,
                          industry_code: str,
                          is_public_company: bool,
                          data_records_at_risk: int = 1000000) -> str:
    """
    Main quotation tool - generates complete premium quote with Tier 1 and Tier 2 coverage
    """
    
    try:
        # Step 1: Predict frequency
        freq_result = predict_frequency(company_revenue_usd, employee_count, is_public_company)
        frequency = freq_result['predicted_frequency']
        
        # Step 2: Predict severity
        sev_result = predict_severity(company_revenue_usd, employee_count, 
                                     is_public_company, data_records_at_risk)
        severity = sev_result['expected_severity']
        
        # Step 3: Get relativities
        industry_rel = get_industry_relativity(industry_code)
        revenue_tier, revenue_rel = get_revenue_tier_relativity(company_revenue_usd)
        
        # Step 4: Calculate premiums
        premium_calc = calculate_pure_premium(frequency, severity)
        base_final_premium = premium_calc['final_premium']
        
        # Step 5: Apply relativities
        adjusted_premium = base_final_premium * industry_rel * revenue_rel
        
        # Step 6: Generate coverage tiers
        coverage_tiers = generate_coverage_tiers(adjusted_premium)
        
        # Build quotation
        quotation = {
            'quotation_id': f"QUOTE-{datetime.now().strftime('%Y%m%d%H%M%S')}",
            'company_information': {
                'name': company_name,
                'revenue': f"${company_revenue_usd/1e9:.1f}B",
                'employees': f"{employee_count:,}",
                'industry': INDUSTRY_CODES.get(industry_code, 'Unknown'),
                'status': 'Public' if is_public_company else 'Private',
                'data_at_risk': f"{data_records_at_risk:,} records"
            },
            'actuarial_analysis': {
                'predicted_frequency': frequency,
                'expected_loss_per_incident': f"${severity:,.0f}",
                'pure_premium': f"${premium_calc['pure_premium']:,.0f}",
                'risk_score': freq_result['risk_score']
            },
            'premium_adjustments': {
                'base_final_premium': f"${base_final_premium:,.0f}",
                'industry_adjustment': f"{industry_rel:.3f}x ({industry_code})",
                'company_size_adjustment': f"{revenue_rel:.3f}x ({revenue_tier})",
                'adjusted_premium': f"${adjusted_premium:,.0f}"
            },
            'loading_breakdown': {
                'acquisition_expenses_20%': f"${premium_calc['loading_components']['acquisition']:,.0f}",
                'administrative_10%': f"${premium_calc['loading_components']['admin']:,.0f}",
                'profit_margin_15%': f"${premium_calc['loading_components']['profit']:,.0f}",
                'uncertainty_buffer_8%': f"${premium_calc['loading_components']['uncertainty']:,.0f}",
                'reinsurance_5%': f"${premium_calc['loading_components']['reinsurance']:,.0f}",
                'total_loading_58%': f"${premium_calc['total_loading']:,.0f}"
            },
            'coverage_options': {
                'tier_1_primary': {
                    'name': coverage_tiers['tier1']['name'],
                    'covers': coverage_tiers['tier1']['attack_vectors'],
                    'annual_premium': f"${coverage_tiers['tier1']['annual_premium']:,.0f}",
                    'description': 'Base cyber insurance coverage for primary threats'
                },
                'tier_2_secondary': {
                    'name': coverage_tiers['tier2']['name'],
                    'covers': coverage_tiers['tier2']['attack_vectors'],
                    'annual_premium': f"${coverage_tiers['tier2']['annual_premium']:,.0f}",
                    'description': 'Extended coverage for emerging and secondary threats',
                    'can_combine': True
                },
                'tier_1_and_2_combined': {
                    'name': coverage_tiers['combined']['name'],
                    'covers': 'All cyber threats',
                    'annual_premium': f"${coverage_tiers['combined']['annual_premium']:,.0f}",
                    'recommendation': coverage_tiers['combined']['recommendation'],
                    'suggested_for': 'Enterprise, Financial Services, Healthcare'
                }
            },
            'next_steps': [
                '1. Review quotation details and coverage options',
                '2. Select preferred coverage tier (Tier 1, Tier 2, or Combined)',
                '3. Request detailed policy terms and conditions',
                '4. Provide any additional underwriting documentation',
                '5. Process application and policy issuance'
            ],
            'notes': [
                'Based on actuarial frequency and severity models',
                'Final premium subject to underwriting approval',
                'Quotation valid for 30 days',
                'Additional discounts available for claims-free experience'
            ]
        }
        
        return json.dumps(quotation, indent=2)
        
    except Exception as e:
        return json.dumps({'error': str(e)}, indent=2)


def explain_coverage_tiers(tier: int) -> str:
    """
    Explain what is covered in each tier
    """
    
    tiers_info = {
        1: {
            'name': 'Tier 1: Primary Coverage',
            'coverage_factors': [
                'Ransomware - Encryption attacks demanding ransom',
                'Data Breaches - Unauthorized access to sensitive data',
                'Supply Chain - Third-party vendor compromise'
            ],
            'includes': [
                'Incident response and forensics',
                'Data recovery and restoration',
                'Legal and regulatory defense',
                'Business interruption',
                'Public relations and notification'
            ],
            'best_for': 'All companies, baseline protection'
        },
        2: {
            'name': 'Tier 2: Secondary Coverage',
            'coverage_factors': [
                'DDoS - Distributed Denial of Service attacks',
                'Malware - Malicious software infections',
                'Trojans - Hidden backdoor access',
                'Backdoors - Persistent unauthorized access',
                'APTs - Advanced Persistent Threats'
            ],
            'includes': [
                'Advanced threat monitoring',
                'Network resilience coverage',
                'Cyber extortion response',
                'Regulatory fines protection',
                'System restoration'
            ],
            'best_for': 'Tech companies, financial services, high-value targets'
        }
    }
    
    if tier not in tiers_info:
        return json.dumps({'error': 'Invalid tier. Please specify 1 or 2.'}, indent=2)
    
    return json.dumps(tiers_info[tier], indent=2)


def compare_coverage_costs(tier1_annual_premium: float) -> str:
    """
    Compare the costs of different coverage combinations
    """
    
    tier2_premium = tier1_annual_premium * 1.25
    combined_premium = tier1_annual_premium + tier2_premium
    
    comparison = {
        'tier_1_only': {
            'annual_cost': f"${tier1_annual_premium:,.0f}",
            'coverage': 'Ransomware, Data Breaches, Supply Chain',
            'best_for': 'Small-medium businesses, traditional industries',
            'percentage_of_combined': f"{(tier1_annual_premium/combined_premium)*100:.1f}%"
        },
        'tier_2_only': {
            'annual_cost': f"${tier2_premium:,.0f}",
            'coverage': 'DDoS, Malware, Trojans, Backdoors, APTs',
            'best_for': 'Tech companies, SaaS providers',
            'percentage_of_combined': f"{(tier2_premium/combined_premium)*100:.1f}%"
        },
        'tier_1_and_2_combined': {
            'annual_cost': f"${combined_premium:,.0f}",
            'coverage': 'All cyber threats (comprehensive)',
            'best_for': 'Enterprise, finance, healthcare',
            'recommendation': 'Most comprehensive protection',
            'savings_vs_separate': f"${0:,.0f} (best value)"  # They're already combined
        }
    }
    
    return json.dumps(comparison, indent=2)

print("✓ Agent tools defined")

## 8. Create the Quotation Agent

Build the Agno agent with Gemini model and tools

In [ ]:
def create_quotation_agent() -> Agent:
    """
    Create and configure the Cyber Risk Premium Quotation Agent
    """
    
    # Define agent tools
    tools = [
        {
            'name': 'premium_quotation_tool',
            'description': 'Generates a complete cyber insurance premium quote. Takes company name, annual revenue (USD), employee count, industry code (51=Tech, 52=Finance, 44-45=Retail, etc.), whether publicly traded, and data records at risk. Returns detailed quotation with Tier 1 (Primary: Ransomware, Data Breaches, Supply Chain) and Tier 2 (Secondary: DDoS, Malware, Trojans, Backdoors, APTs) coverage options.',
            'function': premium_quotation_tool
        },
        {
            'name': 'explain_coverage_tiers',
            'description': 'Explains what is covered in Tier 1 (Primary: Ransomware, Data Breaches, Supply Chain) or Tier 2 (Secondary: DDoS, Malware, Trojans, Backdoors, APTs). Pass tier number 1 or 2.',
            'function': explain_coverage_tiers
        },
        {
            'name': 'compare_coverage_costs',
            'description': 'Compares the annual costs of Tier 1 only, Tier 2 only, or combined Tier 1+2 coverage. Provide the Tier 1 annual premium amount.',
            'function': compare_coverage_costs
        }
    ]
    
    # Create agent with Gemini model
    agent = Agent(
        name='Cyber Risk Premium Quotation Agent',
        model=Gemini(id='gemini-2.0-flash'),
        tools=tools,
        instructions="""You are a professional cyber insurance underwriting agent specializing in premium quotation. Your role is to:

1. GATHER INFORMATION: Collect company details if not provided:
   - Company name
   - Annual revenue (in USD)
   - Number of employees  
   - Industry code (51=Tech, 52=Finance, 44-45=Retail, 92=Telecom, etc.)
   - Public/Private status
   - Data records at risk (optional, default 1M)

2. GENERATE QUOTES: Use the premium_quotation_tool with all information

3. EXPLAIN COVERAGE: 
   - Tier 1 (Primary): Ransomware, Data Breaches, Supply Chain - base premium
   - Tier 2 (Secondary): DDoS, Malware, Trojans, Backdoors, APTs - +25% of base

4. PROVIDE RECOMMENDATIONS:
   - Small/Medium: Tier 1 sufficient
   - Tech/Finance: Consider Tier 1+2
   - Enterprise: Recommend combined coverage

5. ANSWER FOLLOW-UPS about coverage, pricing, or comparisons

TONE: Professional, detailed, actuarially sound. Avoid jargon; explain in business terms.
ACCURACY: Use provided tools; never estimate premiums outside the model.
CLARITY: Show breakdowns of pure premium, loading factors, and coverage comparisons.""",
        markdown=True
    )
    
    return agent


# Create agent instance
quotation_agent = create_quotation_agent()
print("✓ Quotation Agent created successfully")
print(f"  • Model: Gemini 2.0 Flash")
print(f"  • Tools: 3 (quotation, tier explanation, cost comparison)")
print(f"  • Status: Ready for queries")

## 9. Interactive Usage Examples

Example queries to use with the agent

In [ ]:
# Example usage instructions
print("\n" + "="*80)
print("CYBER RISK PREMIUM QUOTATION AGENT - READY FOR USE")
print("="*80)
print("\nThe agent is ready to generate premium quotes and explain coverage options.")
print("\nExample queries:")
print("\n1. Generate a quote for a technology company:")
print('   "I need a cyber insurance quote for TechCorp Inc., a large technology')
print('   company with $150 billion in annual revenue, 250,000 employees, and')
print('   publicly traded. They have about 100 million customer records at risk."')
print("\n2. Ask about coverage differences:")
print('   "What is the difference between Tier 1 and Tier 2 coverage?"')
print("\n3. Compare pricing options:")
print('   "If Tier 1 costs $50M annually, what would Tier 2 and combined cost?"')
print("\n" + "="*80)
print("\nTO USE THE AGENT:")
print("\n  quotation_agent.print_response('Your question here')")
print("\nNOTE: Requires GOOGLE_API_KEY environment variable set.")
print("="*80)

## 10. Run Interactive Demo

Execute example queries (uncomment to run with API key)

In [ ]:
# Example 1: Generate quote for a technology company
query_1 = """
I need a premium quote for a major technology company:
- Name: Silicon Valley Tech Corp
- Revenue: $180 billion
- Employees: 300,000
- Industry: Technology (code 51)
- Status: Publicly traded
- Data at risk: 500 million customer records

Please generate a detailed quote with all coverage options and recommendations.
"""

print("\n" + "="*80)
print("EXAMPLE QUERY 1: Technology Company Quote")
print("="*80)
print(query_1)
print("\n" + "-"*80)
print("Agent Response:")
print("-"*80)
# Uncomment to run with API: quotation_agent.print_response(query_1)
print("(Requires GOOGLE_API_KEY - uncomment line above to execute)")

In [ ]:
# Example 2: Ask about coverage tiers
query_2 = """
We're a financial services company considering cyber insurance. 
Can you explain what Tier 1 and Tier 2 coverage include, and which would be best for us?
"""

print("\n" + "="*80)
print("EXAMPLE QUERY 2: Coverage Explanation")
print("="*80)
print(query_2)
print("\n" + "-"*80)
print("Agent Response:")
print("-"*80)
# Uncomment to run with API: quotation_agent.print_response(query_2)
print("(Requires GOOGLE_API_KEY - uncomment line above to execute)")

In [ ]:
# Example 3: Compare pricing
query_3 = """
If a Tier 1 primary coverage costs $45 million annually, 
how much would Tier 2 secondary coverage cost, and what if we wanted both?
Please compare all options.
"""

print("\n" + "="*80)
print("EXAMPLE QUERY 3: Coverage Cost Comparison")
print("="*80)
print(query_3)
print("\n" + "-"*80)
print("Agent Response:")
print("-"*80)
# Uncomment to run with API: quotation_agent.print_response(query_3)
print("(Requires GOOGLE_API_KEY - uncomment line above to execute)")

## 11. API Setup Instructions

How to use the agent with your Google API key

In [ ]:
print("""
╔════════════════════════════════════════════════════════════════════════════════╗
║                    SETTING UP YOUR GOOGLE GEMINI API KEY                        ║
╚════════════════════════════════════════════════════════════════════════════════╝

1. GET YOUR API KEY:
   → Go to https://aistudio.google.com/apikey
   → Click 'Create API Key'
   → Copy the generated key

2. IN GOOGLE COLAB:
   → Click the secrets icon (🔑) on the left sidebar
   → Click 'Create new secret'
   → Name: GOOGLE_API_KEY
   → Value: (paste your API key)
   → Toggle 'Notebook access' ON

3. IN LOCAL JUPYTER:
   → Set environment variable:
   → export GOOGLE_API_KEY='your-api-key-here'
   → Or use: os.environ['GOOGLE_API_KEY'] = 'your-api-key-here'

4. RUN AGENT QUERIES:
   → quotation_agent.print_response('Your query here')

╔════════════════════════════════════════════════════════════════════════════════╗
║                              READY TO QUOTE!                                    ║
╚════════════════════════════════════════════════════════════════════════════════╝

Agent Features:
  ✓ Frequency prediction (Poisson GLM)
  ✓ Severity prediction (Lognormal)
  ✓ Premium calculation (58% loading)
  ✓ Tier 1 coverage (Ransomware, Breaches, Supply Chain)
  ✓ Tier 2 coverage (DDoS, Malware, Trojans, Backdoors, APTs)
  ✓ Industry adjustments (0.8x - 1.4x)
  ✓ Company size adjustments (0.35x - 2.05x)
  ✓ Natural language understanding

Next Step: Set your GOOGLE_API_KEY and execute:
  quotation_agent.print_response('Generate a quote for...')
""")

## 12. Testing Without API (Local Demo)

Test the core functions without Gemini API

In [ ]:
# Local testing without API - demonstrates the quotation generation
print("\n" + "="*80)
print("LOCAL DEMO: Testing quotation engine (no API needed)")
print("="*80)

# Test case: Large tech company
test_company = {
    'name': 'TechGiant Inc.',
    'revenue': 150e9,  # $150B
    'employees': 250000,
    'industry': '51',  # Technology
    'is_public': True,
    'data_at_risk': 100e6  # 100M records
}

print(f"\nTest Company: {test_company['name']}")
print(f"  Revenue: ${test_company['revenue']/1e9:.0f}B")
print(f"  Employees: {test_company['employees']:,}")
print(f"  Industry: {INDUSTRY_CODES.get(test_company['industry'], 'Unknown')}")
print(f"  Status: {'Public' if test_company['is_public'] else 'Private'}")
print(f"  Data at risk: {test_company['data_at_risk']/1e6:.0f}M records")

# Generate quote
quote_json = premium_quotation_tool(
    test_company['name'],
    test_company['revenue'],
    test_company['employees'],
    test_company['industry'],
    test_company['is_public'],
    int(test_company['data_at_risk'])
)

quote_data = json.loads(quote_json)

print(f"\n" + "-"*80)
print("QUOTATION RESULTS")
print("-"*80)
print(f"\nActuarial Metrics:")
for key, value in quote_data['actuarial_analysis'].items():
    print(f"  {key}: {value}")

print(f"\nPremium Adjustments:")
for key, value in quote_data['premium_adjustments'].items():
    print(f"  {key}: {value}")

print(f"\nCoverage Options:")
print(f"  Tier 1 (Primary): {quote_data['coverage_options']['tier_1_primary']['annual_premium']}")
print(f"    Covers: {', '.join(quote_data['coverage_options']['tier_1_primary']['covers'])}")

print(f"  Tier 2 (Secondary): {quote_data['coverage_options']['tier_2_secondary']['annual_premium']}")
print(f"    Covers: {', '.join(quote_data['coverage_options']['tier_2_secondary']['covers'])}")

print(f"  Combined (Tier 1 + 2): {quote_data['coverage_options']['tier_1_and_2_combined']['annual_premium']}")

print("\n" + "="*80)
print("✓ Local demo complete - ready to use with Gemini API")
print("="*80)